# 11. Lecke - Ügynök-ügynök közötti (A2A) protokoll


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mi az az A2A protokoll?

Az **Agent-to-Agent (A2A) protokoll** egy nyílt szabvány, amely lehetővé teszi az AI ügynökök közötti kommunikációt,
egymás felfedezését és együttműködését – még akkor is, ha különböző keretrendszereken alapulnak vagy különböző szolgáltatóknál vannak
hosztolva.

Kulcsfontosságú fogalmak:

- **Felfedezés** – Az ügynökök közzétesznek egy *Agent Card*-ot, amely leírja képességeiket, így
  más ügynökök (vagy koordinátorok) könnyen megtalálhatják a feladathoz megfelelő szakembert.
- **Üzenetküldés** – Az ügynökök strukturált üzeneteket cserélnek közös protokollon keresztül, így egy
  egyik ügynöktől érkező kérés egy másik ügynök által érthető és teljesíthető, függetlenül annak belső
  megvalósításától.
- **Feladat-életciklus** – Az A2A olyan állapotokat definiál, mint a *beküldött*, *feldolgozás alatt*, *befejezett* és
  *sikertelen*, lehetővé téve a koordinátor számára a delegált feladat állapotának teljes áttekintését.

Ebben a leckében A2A-stílusú együttműködést szimulálunk három specializált utazási ügynök összekapcsolásával
egy olyan munkafolyamatban, ahol minden ügynök a saját szakértelmét adja, és továbbadja az eredményeket a következő ügynöknek.


## Specializált utazási ügynökök létrehozása


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Többügynökös együttműködés munkafolyamat révén

Összekapcsoljuk a három ügynököt egy egymás utáni munkafolyamatba, amely tükrözi az A2A üzenetküldést:

1. **CurrencyExchangeAgent** fogadja a felhasználói kérést, és pénznem-ajánlást készít.
2. **ActivityPlannerAgent** megkapja a kibővített kontextust, és tevékenység-javaslatokat ad hozzá.
3. **TravelManagerAgent** mindkét bemenetet összesíti egy végső utazási összefoglalóba.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Understanding A2A in Production

In a production environment the A2A protocol unlocks powerful cross-service scenarios:

| Capability | Description |
|---|---|
| **Cross-framework interop** | An agent built with one framework can delegate tasks to an agent built with any other A2A-compliant framework, enabling true cross-organization interoperability. |
| **Service boundaries** | Agents can live in separate microservices, cloud regions, or even different organisations while still collaborating seamlessly. |
| **Dynamic discovery** | An orchestrator can query an Agent Card registry at runtime to find the best-suited specialist for a given sub-task. |
| **Streaming & push notifications** | A2A supports Server-Sent Events (SSE) for real-time progress updates and push notifications for long-running tasks. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Összefoglaló

Ebben a leckében megtanultad:

1. **Mi az az A2A protokoll** — egy nyílt szabvány ügynökök közötti felfedezésre, üzenetküldésre,
   és feladatkezelésre.
2. **Hogyan hozz létre specializált ügynököket** — például Pénzváltó ügynököt, Tevékenységtervező ügynököt,
   és Utazáskezelő koordinátort.
3. **Hogyan kapcsolj össze ügynököket egy munkafolyamatba** — a `WorkflowBuilder` használatával, hogy modellezd az ügynökök közti soros
   üzenetküldést.
4. **Hogyan működik az A2A éles környezetben** — lehetővé téve keretrendszerek és szolgáltatások közötti együttműködést
   dinamikus felfedezéssel és folyamatos frissítésekkel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
